# Graph demo: notebooks 03-06 on Taiwan

Runs the graph steps of the pipeline (build graphs, graph features, GNN, hybrid) on the Taiwan
dataset (30k rows) on CPU, using the code in `src/`. Plotting helpers are in `demo_kit.py`.
DGraph-Fin is too large to run here, so its numbers are read from `report/`.

Run all cells in order (about 90 s; the first run also builds the data and a t-SNE map, about 3 min).

In [ ]:
import demo_kit as dk   # import before torch, see note at the top of demo_kit.py
import numpy as np
import pandas as pd
import torch
from torch import nn

dk.style(font_size=18)
D = dk.load("taiwan")

## 03: Build graphs

Taiwan has no real links between customers, so the graph is a kNN graph in feature space.
A degree-preserving random rewire of it is the control.

In [ ]:
from src.graphs import EdgeList, assert_no_test_nodes
from src.graphs import knn as KNN, random_control as RC

ks = D.cfg["graph"]["knn"]["k"]   # [5, 10, 20]
with dk.step("kNN graphs (index over train + val only)"):
    knn_clean, knn_query = KNN.build_knn_all(D.Xnp, D.allowed, D.te, ks, pca_dims=16, seed=0)
with dk.step("random control"):
    rnd_clean = {k: RC.rewire(knn_clean[k], seed=0) for k in ks}
    rnd_query = {k: RC.rewire(knn_query[k], seed=1) for k in ks}

# full graph = clean edges + query edges from test rows into train/val
knn_full = {k: EdgeList.concat(knn_clean[k], knn_query[k]) for k in ks}
rnd_full = {k: EdgeList.concat(rnd_clean[k], rnd_query[k]) for k in ks}

How the builder works, on 36 rows. Click through the steps; the code for each step is shown underneath.

In [ ]:
dk.graph_build_walkthrough(D, k=3)

Leakage check: no graph used for training may touch a test row.

In [ ]:
for k in ks:
    assert_no_test_nodes(knn_clean[k], D.te, f"knn_k{k}")
    assert_no_test_nodes(rnd_clean[k], D.te, f"random_control_k{k}")
print(f"{2 * len(ks)} clean graphs, 0 edges touching the {len(D.te):,} test rows")

# what happens if the index is fitted on all rows
leaky = KNN.KnnIndex(pca_dims=16).fit(D.Xnp, np.arange(D.n), 10).clean_edges([10])[10]
try:
    assert_no_test_nodes(leaky, D.te, "kNN fitted on all rows")
except AssertionError as err:
    print("AssertionError:", err)

Neighbours of a single row. The map is a t-SNE of the space the kNN search runs in.

In [ ]:
dk.neighbourhood_explorer(D, {"kNN": knn_full, "random control": rnd_full})

The rewire keeps each row's out-degree, but GNNs symmetrise the graph first. Mutual kNN pairs
collapse into one edge, so after symmetrising the two graphs no longer have the same degree.

In [ ]:
dk.degree_comparison(D, knn_clean, rnd_clean)

## 04: Tier 1, graph features + XGBoost

Degree, 2-hop reach, neighbour default rate (training labels only) and neighbour feature means,
appended to the tabular features. Same XGBoost settings as the baseline.

In [ ]:
from src.graphs.features import graph_features

cols = list(D.X.columns)
with dk.step("graph features"):
    F_knn = graph_features(knn_full[10], D.Xnp, D.y, D.train_mask, feature_cols=cols)
    F_rnd = graph_features(rnd_full[10], D.Xnp, D.y, D.train_mask, feature_cols=cols)
print(F_knn.shape[1], "graph feature columns")
F_knn.loc[D.te[:5], ["g_degree", "g_two_hop", "g_nb_bad_rate", "g_nbmean_PAY_0", "g_nbdiff_LIMIT_BAL"]].round(2)

In [ ]:
def with_graph(F):
    return pd.concat([D.X.reset_index(drop=True), F.reset_index(drop=True)], axis=1)

race = dk.Race(D, y_range=(0.755, 0.795))
_, _, p_base = race.train("XGBoost", D.X, dk.GREY)
_, _, p_knn = race.train("XGBoost + kNN features", with_graph(F_knn), dk.GOLD)
_, _, p_rnd = race.train("XGBoost + random-control features", with_graph(F_rnd), "#9aa1a8", dash="dot")

In [ ]:
board = dk.Scoreboard(D, "XGBoost", p_base)   # paired bootstrap over the 4,500 test rows
board.add("+ kNN features", p_knn, dk.GOLD)
board.add("+ random-control features", p_rnd, "#9aa1a8");

Same comparison for all four datasets, from the full runs (5 seeds each).

In [ ]:
dk.results_explorer(highlight=("taiwan", "knn"))

## 05: Tier 2, GNN

Two-layer GraphSAGE with mean aggregation. The pipeline uses PyG `SAGEConv` with neighbour sampling
on GPU; this is the same layer written in plain torch and trained full-batch so it runs on CPU.

In [ ]:
class SAGE(nn.Module):
    def __init__(self, d_in, hidden=64, emb=32, dropout=0.3):
        super().__init__()
        self.self1, self.nb1 = nn.Linear(d_in, hidden), nn.Linear(d_in, hidden, bias=False)
        self.self2, self.nb2 = nn.Linear(hidden, emb), nn.Linear(hidden, emb, bias=False)
        self.norm, self.drop, self.head = nn.BatchNorm1d(hidden), nn.Dropout(dropout), nn.Linear(emb, 1)

    def forward(self, x, A):
        # A is row-normalised, so A @ x is the mean over each row's neighbours
        h = self.drop(torch.relu(self.norm(self.self1(x) + self.nb1(A @ x))))
        emb = self.self2(h) + self.nb2(A @ h)
        return self.head(emb).squeeze(-1), emb

In [ ]:
from src.models.prep import TabularPrep

x = torch.tensor(TabularPrep().fit(D.X.iloc[D.tr]).transform(D.X))
A = dk.adjacency(knn_full[10], D.n)
model = SAGE(x.shape[1])
p_sage, emb, snaps, y_snap = dk.train_gnn(model, x, A, D, pace=0.03)

In [ ]:
dk.embedding_animation(snaps, y_snap)

In [ ]:
board.add("GraphSAGE", p_sage[D.te], dk.BLUE)
board.show()

## 06: Tier 3, GNN embedding + XGBoost

The 32-d SAGE embedding is appended to the tabular features and XGBoost is refit.

In [ ]:
from src.models import hybrid as HY

XE = HY.attach_embeddings(D.X, emb, prefix="emb_sage_")
race.show()
booster, _, p_hyb = race.train("XGBoost + SAGE embedding", XE, dk.PURPLE)

In [ ]:
board.add("+ SAGE embedding", p_hyb, dk.PURPLE)
board.show()
dk.attribution_share({"hybrid (SAGE)": HY.contribution_share(booster, XE.iloc[D.va], prefix="emb_sage_")})

## DGraph-Fin

The only dataset with real edges (emergency contacts). Results from the full GPU runs.

In [ ]:
dk.dgraphfin_summary()